# Bank Teller Knowledge Retrieval — Handoff vs Sequential Evaluation

This notebook compares **Sequential** and **Handoff** orchestration patterns for a bank teller knowledge retrieval use case.

**Scenario**: Bank tellers (ICs) and managers search a knowledge base for answers to customer or internal questions. Users have different access levels — managers see financials, HR, and override data that ICs cannot.

| Pattern | How It Works | Role Enforcement |
|---------|-------------|------------------|
| **Sequential** | Query → Analyze → Retrieve → Generate → Format | At retrieval layer (KB filtering) |
| **Handoff** | Query → Router → Specialist (domain-specific) | At routing layer (role-gated routing) |

**Evaluation dimensions**: Latency, accuracy, role adherence, routing precision (handoff only), cost.

---

## 1. Setup & Configuration

In [1]:
import os
import sys
import json
import time
import asyncio
from pathlib import Path
from datetime import datetime

# Ensure backend root is on sys.path (notebook lives inside bank_teller/)
BACKEND_DIR = Path('.').resolve().parent  # go up from bank_teller/ to backend/
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# Install plotly if missing (one-time)
try:
    import plotly
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load environment
from dotenv import load_dotenv
load_dotenv(BACKEND_DIR / '.env', override=True)

# Import project modules
from bank_teller.knowledge_base import (
    KNOWLEDGE_BASE, UserRole, Domain,
    retrieve, format_context, get_domains_for_role,
)
from bank_teller.agents import BankAgentFactory
from bank_teller.sequential_retrieval import run_sequential_retrieval, SequentialResult
from bank_teller.handoff_retrieval import run_handoff_retrieval, HandoffResult
from bank_teller.eval_dataset import (
    EVAL_DATASET, EvalCase,
    get_dataset, get_dataset_by_category, get_dataset_stats,
)

print(f'Backend directory: {BACKEND_DIR}')
print(f'Azure OpenAI endpoint: {os.getenv("AZURE_OPENAI_ENDPOINT", "NOT SET")}')
print(f'Deployment: {os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "NOT SET")}')
print(f'Knowledge base articles: {len(KNOWLEDGE_BASE)}')
print(f'Eval dataset size: {len(EVAL_DATASET)}')
print('\n✅ Setup complete')

C:\Python314\Lib\abc.py:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
  cls = super().__new__(mcls, name, bases, namespace, **kwargs)
C:\Python314\Lib\abc.py:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
  cls = super().__new__(mcls, name, bases, namespace, **kwargs)


Backend directory: C:\Users\gabrielab\projects\agent-patterns-sandbox\patterns\backend
Azure OpenAI endpoint: https://proj-agentorchtest.openai.azure.com/
Deployment: gpt-4o
Knowledge base articles: 16
Eval dataset size: 50

✅ Setup complete


## 2. Knowledge Base Explorer

Let's explore the synthetic banking KB and demonstrate role-based filtering.

In [2]:
# Show all articles with access levels
kb_data = []
for article in KNOWLEDGE_BASE:
    kb_data.append({
        'ID': article.id,
        'Title': article.title,
        'Domain': article.domain.value,
        'Access': '🔒 Manager' if article.min_role == UserRole.MANAGER else '📖 All',
        'Keywords': ', '.join(article.keywords[:4]),
    })

kb_df = pd.DataFrame(kb_data)
print(f'Total articles: {len(kb_df)}')
print(f'IC-accessible: {len(kb_df[kb_df["Access"] == "📖 All"])}')
print(f'Manager-only:  {len(kb_df[kb_df["Access"] == "🔒 Manager"])}\n')
kb_df

Total articles: 16
IC-accessible: 11
Manager-only:  5



,ID,Title,Domain,Access,Keywords
0,acct-001,Personal Checking Account Types,accounts,📖 All,"checking, account types, basic, plus"
1,acct-002,Business Account Opening Requirements,accounts,📖 All,"business, account opening, requirements, EIN"
2,acct-003,Account Closure Procedures,accounts,📖 All,"closure, close account, early closure fee"
3,wire-001,Domestic Wire Transfer Limits and Fees,wire_transfers,📖 All,"wire transfer, domestic, limit, fee"
4,wire-002,International Wire Transfer Procedures,wire_transfers,📖 All,"international, wire, SWIFT, IBAN"
5,disp-001,Debit Card Dispute Process,disputes,📖 All,"dispute, debit card, chargeback, provisional c..."
6,disp-002,ACH Unauthorized Transaction Disputes,disputes,📖 All,"ACH, unauthorized, Reg E, NACHA"
7,lend-001,Personal Loan Products Overview,lending,📖 All,"personal loan, unsecured, secured, HELOC"
8,lend-002,Mortgage Rate Lock Policy,lending,📖 All,"mortgage, rate lock, float-down, points"
9,comp-001,BSA/AML Customer Due Diligence,compliance,📖 All,"BSA, AML, CDD, EDD"


In [3]:
# Demonstrate role-based retrieval filtering
test_query = "What's the branch P&L and revenue?"

print(f'Query: "{test_query}"\n')
print('--- IC Results ---')
ic_results = retrieve(test_query, role=UserRole.IC)
for r in ic_results:
    print(f'  {r.id}: {r.title}')
if not ic_results:
    print('  (no results — financial data is manager-only)')

print('\n--- Manager Results ---')
mgr_results = retrieve(test_query, role=UserRole.MANAGER)
for r in mgr_results:
    print(f'  {r.id}: {r.title}')

Query: "What's the branch P&L and revenue?"

--- IC Results ---
  (no results — financial data is manager-only)

--- Manager Results ---
  fin-001: Branch Q1 2025 P&L Summary
  fin-002: Branch Fee Revenue Breakdown
  hr-001: Branch Staffing and Headcount


## 3. Evaluation Dataset Overview

In [4]:
stats = get_dataset_stats()
print(f'Total test cases: {stats["total"]}\n')

print('By Category:')
for cat, count in stats['by_category'].items():
    print(f'  {cat}: {count}')

print(f'\nBy Role:')
for role, count in stats['by_role'].items():
    print(f'  {role}: {count}')

print(f'\nBy Complexity:')
for comp, count in stats['by_complexity'].items():
    print(f'  {comp}: {count}')

# Show sample cases
eval_df = pd.DataFrame([{
    'ID': c.id,
    'Query': c.query[:60] + ('...' if len(c.query) > 60 else ''),
    'Role': c.user_role.value,
    'Expected Domain': c.expected_domain,
    'Complexity': c.complexity,
    'Category': c.category,
} for c in EVAL_DATASET])

eval_df

Total test cases: 50

By Category:
  simple_lookup: 15
  role_gated: 10
  multi_domain: 10
  compliance: 10
  edge_case: 5

By Role:
  ic: 42
  manager: 8

By Complexity:
  simple: 26
  moderate: 11
  complex: 13


,ID,Query,Role,Expected Domain,Complexity,Category
0,simple-01,What's the daily wire transfer limit for perso...,ic,policy,simple,simple_lookup
1,simple-02,How much does it cost to send a domestic wire?,ic,policy,simple,simple_lookup
2,simple-03,What are the checking account tiers we offer?,ic,policy,simple,simple_lookup
3,simple-04,What documents do I need to open a business ch...,ic,policy,simple,simple_lookup
4,simple-05,How long does it take to process an internatio...,ic,policy,simple,simple_lookup
5,simple-06,What's the minimum credit score for an unsecur...,ic,lending,simple,simple_lookup
6,simple-07,What is the rate lock policy for mortgages?,ic,lending,simple,simple_lookup
7,simple-08,Is there an early closure fee for checking acc...,ic,policy,simple,simple_lookup
8,simple-09,How do I close a joint checking account?,ic,policy,simple,simple_lookup
9,simple-10,What's the APR range for unsecured personal lo...,ic,lending,simple,simple_lookup


## 4. Sequential Pattern Demo

Run 3 example queries through the Sequential pipeline to see how it works.

In [5]:
demo_cases = [
    ('What is the daily wire transfer limit for personal accounts?', UserRole.IC),
    ('Show me the branch P&L', UserRole.IC),          # role-gated
    ('Show me the branch P&L', UserRole.MANAGER),     # same query, different role
]

for query, role in demo_cases:
    print(f'\n{"="*70}')
    print(f'Query: "{query}"  |  Role: {role.value.upper()}')
    print('='*70)
    
    result = await run_sequential_retrieval(query, role=role)
    
    print(f'\n📊 Articles found: {result.articles_found}')
    print(f'⏱️  Latency: {result.latency_seconds:.1f}s')
    print(f'\n💬 Answer:\n{result.answer[:500]}')
    
    time.sleep(2)  # rate limit safety


Query: "What is the daily wire transfer limit for personal accounts?"  |  Role: IC

📊 Articles found: 3
⏱️  Latency: 18.5s

💬 Answer:
📋 ANSWER: 
- Domestic wire transfer limit for personal accounts: **$10,000/day**.
- International wire transfer limit for personal accounts: **$25,000/day**.

⚠️ IMPORTANT: 
- Domestic wires over $10,000 trigger **CTR (Currency Transaction Report) filing** automatically.
- International wires are subject to OFAC screening, and wires to sanctioned countries are blocked.

📞 CUSTOMER SCRIPT: 
"For domestic wires, the daily limit for personal accounts is $10,000. For international wires, the limit i

Query: "Show me the branch P&L"  |  Role: IC

📊 Articles found: 2
⏱️  Latency: 14.8s

💬 Answer:
📋 ANSWER: The request for branch Profit & Loss (P&L) cannot be addressed using the provided context, as the documentation does not cover P&L information or access procedures.

⚠️ IMPORTANT: 
- Branch P&L data is typically restricted and accessible only by managers or

## 5. Handoff Pattern Demo

Run the same 3 queries through the Handoff pipeline — notice routing decisions and role gating.

In [6]:
for query, role in demo_cases:
    print(f'\n{"="*70}')
    print(f'Query: "{query}"  |  Role: {role.value.upper()}')
    print('='*70)
    
    result = await run_handoff_retrieval(query, role=role)
    
    if result.routing_decision:
        rd = result.routing_decision
        print(f'\n🔀 Routing: {rd["specialist"]} (confidence: {rd["confidence"]:.0%})')
        print(f'   Reason: {rd["reasoning"]}')
    
    if result.was_role_blocked:
        print('🚫 ROLE BLOCKED')
    
    print(f'\n📊 Articles found: {result.articles_found}')
    print(f'⏱️  Latency: {result.latency_seconds:.1f}s')
    print(f'\n💬 Answer:\n{result.answer[:500]}')
    
    time.sleep(2)


Query: "What is the daily wire transfer limit for personal accounts?"  |  Role: IC

🔀 Routing: policy (confidence: 100%)
   Reason: The query pertains to account policies regarding wire transfer limits, which falls under the 'policy' specialist's domain.

📊 Articles found: 3
⏱️  Latency: 10.3s

💬 Answer:
The daily wire transfer limit for personal accounts is $10,000 for domestic wires. For international wires, the limit is $25,000/day.

Query: "Show me the branch P&L"  |  Role: IC

🔀 Routing: manager_insights (confidence: 100%)
   Reason: Branch P&L falls under managerial insights, which is outside the access scope of an IC (individual contributor) role.
🚫 ROLE BLOCKED

📊 Articles found: 0
⏱️  Latency: 6.3s

💬 Answer:
I'm sorry, but that information is only available to managers. The data you're requesting (branch financials, staffing, or override authorities) requires manager-level access. Please ask your branch manager for assistance.

Query: "Show me the branch P&L"  |  Role: MANAG

## 6. Phase 1 — Collect Responses (Full Evaluation Run)

Run all 50 test cases through both patterns. Results are saved incrementally to JSONL for resumability.

In [7]:
RESULTS_DIR = BACKEND_DIR / 'bank_teller' / 'eval_results'
RESULTS_DIR.mkdir(exist_ok=True)

RESULTS_FILE = RESULTS_DIR / 'eval_responses.jsonl'

# Load existing results for resumability
completed = set()
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            completed.add(f"{row['case_id']}_{row['pattern']}")
    print(f'Resuming: {len(completed)} runs already completed')
else:
    print('Starting fresh evaluation run')

total_runs = len(EVAL_DATASET) * 2  # 2 patterns
remaining = total_runs - len(completed)
print(f'Total runs: {total_runs}, Remaining: {remaining}')

Starting fresh evaluation run
Total runs: 100, Remaining: 100


In [ ]:
# Run evaluation — resumable
run_count = 0
errors = []

for case in EVAL_DATASET:
    for pattern_name in ['sequential', 'handoff']:
        run_key = f'{case.id}_{pattern_name}'
        if run_key in completed:
            continue
        
        run_count += 1
        print(f'[{run_count}/{remaining}] {pattern_name.upper():10s} | {case.id:10s} | {case.query[:50]}...')
        
        try:
            if pattern_name == 'sequential':
                result = await run_sequential_retrieval(case.query, role=case.user_role)
                row = {
                    'case_id': case.id,
                    'pattern': 'sequential',
                    'query': case.query,
                    'user_role': case.user_role.value,
                    'response': result.answer,
                    'context': result.context_used,
                    'ground_truth': case.ground_truth,
                    'expected_domain': case.expected_domain,
                    'category': case.category,
                    'complexity': case.complexity,
                    'latency_seconds': result.latency_seconds,
                    'articles_found': result.articles_found,
                    'specialist_used': '',
                    'routing_confidence': None,
                    'was_role_blocked': False,
                }
            else:
                result = await run_handoff_retrieval(case.query, role=case.user_role)
                row = {
                    'case_id': case.id,
                    'pattern': 'handoff',
                    'query': case.query,
                    'user_role': case.user_role.value,
                    'response': result.answer,
                    'context': result.context_used,
                    'ground_truth': case.ground_truth,
                    'expected_domain': case.expected_domain,
                    'category': case.category,
                    'complexity': case.complexity,
                    'latency_seconds': result.latency_seconds,
                    'articles_found': result.articles_found,
                    'specialist_used': result.specialist_used,
                    'routing_confidence': result.routing_decision.get('confidence') if result.routing_decision else None,
                    'was_role_blocked': result.was_role_blocked,
                }
            
            # Write incrementally
            with open(RESULTS_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
            
            completed.add(run_key)
            
        except Exception as e:
            print(f'  ❌ ERROR: {e}')
            errors.append({'case_id': case.id, 'pattern': pattern_name, 'error': str(e)})
        
        time.sleep(2)  # rate limit safety

print(f'\n✅ Evaluation complete. {len(completed)} total runs. {len(errors)} errors.')
if errors:
    print('\nErrors:')
    for e in errors:
        print(f'  {e["case_id"]} ({e["pattern"]}): {e["error"]}')

[1/100] SEQUENTIAL | simple-01  | What's the daily wire transfer limit for personal ...


## 7. Phase 1.5 — Simple Metrics Analysis

Analyze results using straightforward metrics: latency, token proxy (article count), routing precision, role adherence.

In [ ]:
# Load all results
results = []
with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        results.append(json.loads(line))

df = pd.DataFrame(results)
print(f'Loaded {len(df)} evaluation results')
print(f'Patterns: {df["pattern"].unique()}')
print(f'Roles: {df["user_role"].unique()}')
df.head()

In [ ]:
# --- Latency Comparison ---
latency_summary = df.groupby('pattern')['latency_seconds'].agg(['mean', 'median', 'std', 'min', 'max'])
print('⏱️  Latency Summary (seconds):')
print(latency_summary.round(2))

fig = px.box(
    df, x='pattern', y='latency_seconds', color='pattern',
    title='Latency Distribution: Sequential vs Handoff',
    labels={'latency_seconds': 'Latency (seconds)', 'pattern': 'Pattern'},
    color_discrete_map={'sequential': '#636EFA', 'handoff': '#EF553B'},
)
fig.show()

In [ ]:
# --- Latency by Category ---
fig = px.box(
    df, x='category', y='latency_seconds', color='pattern',
    title='Latency by Query Category',
    labels={'latency_seconds': 'Latency (seconds)', 'category': 'Category'},
    color_discrete_map={'sequential': '#636EFA', 'handoff': '#EF553B'},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# --- Role Adherence Analysis ---
# For role-gated queries: did the system correctly block IC access to manager data?
role_gated = df[df['category'] == 'role_gated'].copy()

# IC users asking for manager-only data should get blocked
ic_manager_queries = role_gated[role_gated['user_role'] == 'ic']

print('🔒 Role Adherence — IC users querying manager-only data:\n')

for pattern in ['sequential', 'handoff']:
    pattern_data = ic_manager_queries[ic_manager_queries['pattern'] == pattern]
    if len(pattern_data) == 0:
        continue
    
    # Check if response indicates access denial
    blocked_keywords = ['denied', 'manager', 'not available', 'cannot', 'restricted', 'access', 'supervisor']
    blocked_count = sum(
        any(kw in str(row['response']).lower() for kw in blocked_keywords)
        for _, row in pattern_data.iterrows()
    )
    total = len(pattern_data)
    print(f'  {pattern.upper():12s}: {blocked_count}/{total} correctly blocked ({blocked_count/total*100:.0f}%)')

# Handoff-specific: check was_role_blocked flag
handoff_blocked = ic_manager_queries[ic_manager_queries['pattern'] == 'handoff']
if len(handoff_blocked) > 0:
    explicit_blocks = handoff_blocked['was_role_blocked'].sum()
    print(f'\n  Handoff explicit role blocks: {explicit_blocks}/{len(handoff_blocked)}')

In [ ]:
# --- Routing Precision (Handoff only) ---
handoff_df = df[df['pattern'] == 'handoff'].copy()

# Compare specialist_used vs expected_domain
handoff_df['routing_correct'] = handoff_df.apply(
    lambda row: row['specialist_used'] == row['expected_domain'] 
    if row['specialist_used'] else False,
    axis=1
)

routing_accuracy = handoff_df['routing_correct'].mean()
print(f'🎯 Handoff Routing Precision: {routing_accuracy:.1%}\n')

# Breakdown by category
routing_by_cat = handoff_df.groupby('category')['routing_correct'].agg(['sum', 'count', 'mean'])
routing_by_cat.columns = ['Correct', 'Total', 'Accuracy']
print(routing_by_cat.round(2))

# Routing confidence distribution
if handoff_df['routing_confidence'].notna().any():
    fig = px.histogram(
        handoff_df[handoff_df['routing_confidence'].notna()],
        x='routing_confidence',
        title='Handoff Router Confidence Distribution',
        nbins=20,
        labels={'routing_confidence': 'Confidence Score'},
    )
    fig.show()

In [ ]:
# --- Summary Comparison Table ---
summary = df.groupby('pattern').agg(
    avg_latency=('latency_seconds', 'mean'),
    p95_latency=('latency_seconds', lambda x: x.quantile(0.95)),
    avg_articles=('articles_found', 'mean'),
    total_runs=('case_id', 'count'),
).round(2)

print('📊 Pattern Comparison Summary')
print('=' * 60)
print(summary)

# Add routing precision for handoff
if len(handoff_df) > 0:
    print(f'\nHandoff routing precision: {routing_accuracy:.1%}')
    avg_confidence = handoff_df['routing_confidence'].mean()
    if pd.notna(avg_confidence):
        print(f'Handoff avg routing confidence: {avg_confidence:.1%}')

## 8. Phase 2 — AI-Judge Evaluators (Optional)

Uses Azure AI Evaluation SDK to run quality evaluators on the collected responses.

**Prerequisites**: `azure-ai-evaluation` package and Azure AI Project endpoint configured.

Skip this section if you only want simple metrics.

In [ ]:
# Check if azure-ai-evaluation is available
AI_EVAL_AVAILABLE = False
try:
    from azure.ai.evaluation import (
        evaluate,
        GroundednessEvaluator,
        RelevanceEvaluator,
        CoherenceEvaluator,
        FluencyEvaluator,
    )
    AI_EVAL_AVAILABLE = True
    print('✅ azure-ai-evaluation SDK available')
except ImportError:
    print('⚠️  azure-ai-evaluation not installed. Skipping AI-judge evaluators.')
    print('   Install with: pip install azure-ai-evaluation')

In [ ]:
if AI_EVAL_AVAILABLE:
    from azure.identity import DefaultAzureCredential
    
    # Reduce concurrency to avoid 429s
    os.environ['PF_WORKER_COUNT'] = '2'
    
    project_endpoint = os.getenv('AZURE_AI_PROJECT_ENDPOINT')
    if not project_endpoint:
        print('⚠️  AZURE_AI_PROJECT_ENDPOINT not set. Cannot run AI evaluators.')
        AI_EVAL_AVAILABLE = False
    else:
        credential = DefaultAzureCredential()
        model_config = {
            'azure_endpoint': os.getenv('AZURE_OPENAI_ENDPOINT'),
            'azure_deployment': os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
            'api_version': os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21'),
        }
        print(f'AI Eval configured: {project_endpoint}')

In [ ]:
if AI_EVAL_AVAILABLE:
    # Batch 1: Quality evaluators
    print('Running quality evaluators (Groundedness, Relevance, Coherence, Fluency)...')
    
    quality_results = evaluate(
        data=str(RESULTS_FILE),
        evaluators={
            'groundedness': GroundednessEvaluator(model_config=model_config, credential=credential),
            'relevance': RelevanceEvaluator(model_config=model_config, credential=credential),
            'coherence': CoherenceEvaluator(model_config=model_config, credential=credential),
            'fluency': FluencyEvaluator(model_config=model_config, credential=credential),
        },
        evaluator_config={
            'groundedness': {'query': '${data.query}', 'response': '${data.response}', 'context': '${data.context}'},
            'relevance': {'query': '${data.query}', 'response': '${data.response}', 'context': '${data.context}'},
            'coherence': {'query': '${data.query}', 'response': '${data.response}'},
            'fluency': {'query': '${data.query}', 'response': '${data.response}'},
        },
    )
    
    print('\n✅ Quality evaluation complete')
    
    # Save quality results
    quality_file = RESULTS_DIR / 'quality_eval_results.json'
    with open(quality_file, 'w', encoding='utf-8') as f:
        json.dump(quality_results.get('metrics', {}), f, indent=2)
    print(f'Saved to {quality_file}')
    
    # Display summary
    if 'metrics' in quality_results:
        print('\n📊 Quality Metrics:')
        for metric, value in quality_results['metrics'].items():
            print(f'  {metric}: {value}')
else:
    print('Skipping AI-judge evaluators (azure-ai-evaluation not available or not configured)')

In [ ]:
if AI_EVAL_AVAILABLE:
    # Cooldown between batches
    print('Waiting 30s between evaluator batches...')
    time.sleep(30)
    
    # Batch 2: Agentic evaluators
    from azure.ai.evaluation import TaskAdherenceEvaluator, ResponseCompletenessEvaluator
    
    print('Running agentic evaluators (TaskAdherence, ResponseCompleteness)...')
    
    agentic_results = evaluate(
        data=str(RESULTS_FILE),
        evaluators={
            'task_adherence': TaskAdherenceEvaluator(model_config=model_config, credential=credential),
            'response_completeness': ResponseCompletenessEvaluator(model_config=model_config, credential=credential),
        },
        evaluator_config={
            'task_adherence': {'query': '${data.query}', 'response': '${data.response}'},
            'response_completeness': {'response': '${data.response}', 'ground_truth': '${data.ground_truth}'},
        },
    )
    
    print('\n✅ Agentic evaluation complete')
    
    agentic_file = RESULTS_DIR / 'agentic_eval_results.json'
    with open(agentic_file, 'w', encoding='utf-8') as f:
        json.dump(agentic_results.get('metrics', {}), f, indent=2)
    
    if 'metrics' in agentic_results:
        print('\n📊 Agentic Metrics:')
        for metric, value in agentic_results['metrics'].items():
            print(f'  {metric}: {value}')

## 9. Results Dashboard

In [ ]:
# --- Comprehensive Comparison Dashboard ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Avg Latency by Category',
        'Articles Retrieved',
        'Latency Distribution',
        'Handoff Routing Confidence',
    ),
)

# 1. Avg latency by category
lat_by_cat = df.groupby(['category', 'pattern'])['latency_seconds'].mean().reset_index()
for pattern in ['sequential', 'handoff']:
    data = lat_by_cat[lat_by_cat['pattern'] == pattern]
    fig.add_trace(
        go.Bar(x=data['category'], y=data['latency_seconds'], name=f'{pattern} latency'),
        row=1, col=1,
    )

# 2. Articles retrieved
art_by_pat = df.groupby(['pattern', 'category'])['articles_found'].mean().reset_index()
for pattern in ['sequential', 'handoff']:
    data = art_by_pat[art_by_pat['pattern'] == pattern]
    fig.add_trace(
        go.Bar(x=data['category'], y=data['articles_found'], name=f'{pattern} articles'),
        row=1, col=2,
    )

# 3. Latency distribution
for pattern in ['sequential', 'handoff']:
    data = df[df['pattern'] == pattern]
    fig.add_trace(
        go.Histogram(x=data['latency_seconds'], name=f'{pattern}', opacity=0.7),
        row=2, col=1,
    )

# 4. Routing confidence (handoff only)
conf_data = handoff_df[handoff_df['routing_confidence'].notna()]
if len(conf_data) > 0:
    fig.add_trace(
        go.Histogram(x=conf_data['routing_confidence'], name='confidence', nbinsx=20),
        row=2, col=2,
    )

fig.update_layout(
    height=700,
    title_text='Bank Teller Eval: Sequential vs Handoff',
    showlegend=True,
)
fig.show()

## 10. Recommendation & Decision Framework

Based on the evaluation results, here's when to use each pattern.

In [ ]:
# Generate recommendation based on results
seq_latency = df[df['pattern'] == 'sequential']['latency_seconds'].mean()
ho_latency = df[df['pattern'] == 'handoff']['latency_seconds'].mean()
latency_delta = ho_latency - seq_latency

# Role adherence (for role-gated IC queries)
ic_manager = df[(df['category'] == 'role_gated') & (df['user_role'] == 'ic')]
blocked_keywords = ['denied', 'manager', 'not available', 'cannot', 'restricted', 'access', 'supervisor']

seq_role_data = ic_manager[ic_manager['pattern'] == 'sequential']
ho_role_data = ic_manager[ic_manager['pattern'] == 'handoff']

seq_blocked = sum(
    any(kw in str(r['response']).lower() for kw in blocked_keywords)
    for _, r in seq_role_data.iterrows()
) if len(seq_role_data) > 0 else 0

ho_blocked = sum(
    any(kw in str(r['response']).lower() for kw in blocked_keywords)
    for _, r in ho_role_data.iterrows()
) if len(ho_role_data) > 0 else 0

seq_role_pct = seq_blocked / max(len(seq_role_data), 1) * 100
ho_role_pct = ho_blocked / max(len(ho_role_data), 1) * 100

print('=' * 70)
print('📋 EVALUATION RESULTS & RECOMMENDATION')
print('=' * 70)
print(f'''
┌──────────────────────┬──────────────┬──────────────┐
│ Metric               │ Sequential   │ Handoff      │
├──────────────────────┼──────────────┼──────────────┤
│ Avg Latency          │ {seq_latency:>8.1f}s    │ {ho_latency:>8.1f}s    │
│ Latency Delta        │     —        │ +{latency_delta:>6.1f}s    │
│ Role Adherence (IC)  │ {seq_role_pct:>8.0f}%    │ {ho_role_pct:>8.0f}%    │
│ Routing Precision    │     N/A      │ {routing_accuracy:>8.1%}    │
│ Architecture         │ Fixed pipe   │ Dynamic route│
└──────────────────────┴──────────────┴──────────────┘
''')

# Decision
print('🏆 RECOMMENDATION:')
if ho_role_pct > seq_role_pct and routing_accuracy > 0.85:
    print('  → Use HANDOFF for production.')
    print('    ✅ Better role enforcement at the routing layer')
    print('    ✅ Domain-specialized agents give more precise answers')
    print(f'    ⚠️  +{latency_delta:.1f}s latency overhead (router step)')
    print('    💡 Acceptable for bank tellers who need accuracy over speed')
elif seq_role_pct >= ho_role_pct:
    print('  → Start with SEQUENTIAL, add Handoff when role complexity grows.')
    print('    ✅ Simpler architecture, lower latency')
    print('    ✅ Adequate role filtering at retrieval layer')
    print('    ⚠️  May struggle as domain specialization needs grow')
else:
    print('  → Use HANDOFF with tuning.')
    print(f'    ⚠️  Routing precision ({routing_accuracy:.0%}) needs improvement')
    print('    💡 Refine router instructions or add more training examples')

print(f'''
\n📌 DECISION FRAMEWORK:
  • >80% simple lookups, minimal role gating → Sequential
  • Mixed query types + role enforcement critical → Handoff
  • Need to add new domains over time → Handoff (just add specialist)
  • Latency-critical, high-volume → Sequential (or cache common queries)
''')